<a href="https://colab.research.google.com/github/SindyBatzz/semana6-segmentacion/blob/main/Semana_6/notebook_segmentacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TAREA · Segmentación: lo que Orange no trae
Dos técnicas: **mezclas gaussianas (GMM)** y **k-prototypes** (datos mixtos). Ejecute las celdas en orden y responda las preguntas. Los archivos `clientes_retail.csv` y `formas_alargadas.csv` deben estar en la misma carpeta que el notebook.

Al final guardará sus resultados (`segmentos_notebook.csv` y los modelos en `.joblib`) para usarlos en su app de Streamlit.

In [3]:
!pip install kmodes -q
import pandas as pd, numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import adjusted_rand_score
from kmodes.kprototypes import KPrototypes

## 1. Mezclas gaussianas: grupos alargados
k-means asume grupos redondos. GMM permite grupos con forma de elipse inclinada. Los datos son sintéticos y tenemos la respuesta real (`grupo_real`).

In [4]:
!wget -q -O clientes_retail.csv https://raw.githubusercontent.com/SindyBatzz/semana6-segmentacion/main/Semana_6/clientes_retail.csv
!wget -q -O formas_alargadas.csv https://raw.githubusercontent.com/SindyBatzz/semana6-segmentacion/main/Semana_6/formas_alargadas.csv
!ls *.csv
a = pd.read_csv('formas_alargadas.csv')
a = a.rename(columns={'m#grupo_real':'grupo_real'})
Z = StandardScaler().fit_transform(a[['x','y']])
km = KMeans(3, n_init=10, random_state=0).fit_predict(Z)
gm = GaussianMixture(3, random_state=0, n_init=5).fit_predict(Z)
print('Acuerdo con la verdad (1 = perfecto)')
print('k-means:', round(adjusted_rand_score(a.grupo_real, km), 2))
print('GMM    :', round(adjusted_rand_score(a.grupo_real, gm), 2))

clientes_retail.csv  formas_alargadas.csv
Acuerdo con la verdad (1 = perfecto)
k-means: 0.82
GMM    : 0.97


**Pregunta:** ¿por qué GMM acierta más? ¿Qué ventaja extra tiene (pista: da una probabilidad de pertenecer a cada grupo)?

**Respuesta N1.** GMM acierta más (acuerdo 0.97 contra 0.82 de k-means) porque cada grupo se modela como una elipse con su propia forma, tamaño e inclinación (su matriz de covarianza). k-means asigna cada punto al centro más cercano. Eso supone grupos redondos y del mismo tamaño, y traza fronteras rectas a medio camino entre centros. En grupos alargados e inclinados, la punta de un grupo queda más cerca del centro del vecino, y k-means la asigna mal.

La ventaja extra es que GMM no solo dice a qué grupo pertenece cada punto: también da la **probabilidad de pertenecer a cada grupo**. Con eso sabemos qué tan seguros estamos de cada asignación y podemos distinguir a los clientes claramente de un segmento de los que están en la frontera.

In [5]:
probs = GaussianMixture(3, random_state=0, n_init=5).fit(Z).predict_proba(Z)
print('Clientes con duda (ninguna probabilidad > 0.9):', int((probs.max(axis=1) < 0.9).sum()), 'de', len(a))

Clientes con duda (ninguna probabilidad > 0.9): 5 de 450


**Respuesta N2.** Quedaron **5 de 450 puntos con duda** (ninguna probabilidad mayor a 0.9), todos en las zonas donde dos elipses se tocan.

Si fueran clientes reales, marketing no debería forzarlos a un solo grupo:
- tratarlos como un grupo "en transición" y vigilarlos, porque son los que tienen más probabilidad de cambiar de segmento;
- usar sus probabilidades para decidir: por ejemplo, darles la oferta del segmento con mayor probabilidad, o probar los dos mensajes (prueba A/B) y quedarse con el que responda mejor;
- no gastar en ellos el presupuesto del segmento más caro hasta confirmar a qué grupo pertenecen.

## 2. K-prototypes: números y categorías juntos
k-means solo entiende números. K-prototypes combina distancia numérica con coincidencia de categorías (aquí `educacion` y `estado_civil`).

In [6]:
c = pd.read_csv('clientes_retail.csv')
c = c.rename(columns={'m#id_cliente':'id_cliente','D#educacion':'educacion','D#estado_civil':'estado_civil','mD#respondio_ultima_campana':'respondio'})
num = ['ingreso_anual','gasto_total','compras_tienda','compras_web','compras_catalogo','visitas_web_mes','recencia_dias','hijos_pequenos','edad']
cat = ['educacion','estado_civil']
X = pd.concat([pd.DataFrame(StandardScaler().fit_transform(c[num]), columns=num), c[cat].reset_index(drop=True)], axis=1)
kp = KPrototypes(n_clusters=3, init='Cao', random_state=0, n_init=3)
c['seg_kproto'] = kp.fit_predict(X.values, categorical=[len(num), len(num)+1])
print(c.groupby('seg_kproto')[['ingreso_anual','gasto_total','hijos_pequenos','edad']].mean().round(1))
print(c.seg_kproto.value_counts())

            ingreso_anual  gasto_total  hijos_pequenos  edad
seg_kproto                                                  
0                 58018.6        669.5             0.2  55.7
1                 76080.7       1298.4             0.0  51.9
2                 33979.6         85.2             0.8  47.8
seg_kproto
2    1012
0     627
1     566
Name: count, dtype: int64


## 3. ¿Coinciden con k-means?

In [7]:
km3 = KMeans(3, n_init=10, random_state=0).fit_predict(StandardScaler().fit_transform(c[num]))
print('Acuerdo k-means vs k-prototypes:', round(adjusted_rand_score(km3, c.seg_kproto), 2))
print(pd.crosstab(km3, c.seg_kproto, rownames=['k-means'], colnames=['k-prototypes']))

Acuerdo k-means vs k-prototypes: 0.99
k-prototypes    0    1     2
k-means                     
0               1  560     0
1               1    0  1012
2             625    6     0


**Respuesta N3.** No: agregar educación y estado civil **casi no cambió los grupos**. El acuerdo entre k-means y k-prototypes es 0.99 y solo unos 8 de 2,205 clientes cambian de grupo. El gasto medio por segmento tampoco cambia:

| Segmento | k-means: clientes / gasto medio | k-prototypes: clientes / gasto medio |
|---|---|---|
| Premium consolidados | 561 / US\$1,304 | 566 / US\$1,298 |
| Compradores digitales de gasto medio | 631 / US\$670 | 627 / US\$670 |
| Familias de bajo gasto | 1,013 / US\$85 | 1,012 / US\$85 |

Esto tiene dos explicaciones. Primero, las dos categorías se reparten casi igual en los tres grupos: alrededor de la mitad de cada segmento tiene licenciatura y cerca del 38% está casado, así que no ayudan a distinguir clientes. Segundo, k-prototypes le dio a las categorías un peso bajo (gamma = 0.5) frente a nueve variables numéricas.

Como dato adicional, GMM sobre los clientes reales (sección 4) sí da grupos distintos (acuerdo con k-means ≈ 0.32): separa principalmente por tener o no hijos pequeños, deja un grupo de ~1,270 clientes sin hijos pequeños que concentra el 86% del gasto, y tiene peor silueta (0.19 contra 0.25). Es menos útil para diferenciar campañas.

**Preguntas finales:** ¿qué segmentación le daría a marketing y por qué? ¿Cambia el valor por segmento (gasto medio) según la técnica?

**Respuesta N4.**
1. Le daríamos a marketing la segmentación de **k-means con k = 3** sobre las variables de comportamiento estandarizadas: premium consolidados, compradores digitales de gasto medio y familias de bajo gasto.
2. Es la técnica más simple de explicar y de recalcular, es estable, y k-prototypes llega prácticamente a los mismos grupos. Agregar educación y estado civil no aporta nada y suma datos personales.
3. Acción: **retener** al segmento premium (25% de los clientes, 59% del gasto y 25% de respuesta a campañas) con un programa de lealtad y beneficios exclusivos, y **crecer** al segmento medio con venta cruzada por web.

## 4. Guarde sus resultados
Se guarda la asignación de cada cliente (para el visor de la app) y los modelos ajustados con `joblib`.

In [8]:
import joblib
sc = StandardScaler().fit(c[num])
Zc = sc.transform(c[num])
gm_retail = GaussianMixture(3, random_state=0, n_init=5).fit(Zc)
c['seg_gmm'] = gm_retail.predict(Zc)
c['seg_kmeans'] = km3
# Cluster = la técnica que usted recomienda (cámbiela si prefiere otra)
# Recomendamos k-means (k = 3); k-prototypes da prácticamente los mismos grupos
c['Cluster'] = 'C' + (c['seg_kmeans'] + 1).astype(str)
c.to_csv('segmentos_notebook.csv', index=False)
joblib.dump({'scaler': sc, 'modelo': gm_retail, 'variables': num}, 'modelo_gmm.joblib')
joblib.dump({'modelo': kp, 'variables_numericas': num, 'variables_categoricas': cat}, 'modelo_kproto.joblib')
print('Guardado: segmentos_notebook.csv, modelo_gmm.joblib, modelo_kproto.joblib')

Guardado: segmentos_notebook.csv, modelo_gmm.joblib, modelo_kproto.joblib
